# 11. Master Annotation Audit & Correction

## Problem

The `_det` (detected) column in the master annotation can be `1` for a species
even when `_orth` (orthology / liftback) is `0`. This is biologically incoherent:
if the orthologous region doesn't exist in a species, we cannot meaningfully say
the peak was "detected" there.

## Root cause: unidirectional liftover

Our pipeline works as follows:

1. Map reads to each species' native genome → call peaks per species
2. Merge peaks into per-species consensus sets (500 bp)
3. **Lift non-human peaks → hg38** (species→hg38)
4. Merge all lifted peaks + human peaks into unified hg38 consensus (500 bp)
   - The `species_detected` column tracks which species contributed
5. **Lift consensus back** to each species (hg38→species)
   - `04b`: Filter by size (reject peaks that changed size)
   - `04c`: Reciprocal check (lift back to hg38 again, check same location)
6. `_orth` = liftback succeeded → orthologous region exists
7. `_det` = species appeared in `species_detected` column

The mismatch arises because the **forward lift** (species→hg38) can succeed
while the **reverse lift** (hg38→species) fails. This happens when:
- The species peak lifted to a hg38 region that overlaps the consensus window
- But the consensus window itself (potentially shifted/merged) cannot be cleanly
  mapped back to the species genome

These are **unidirectional liftover** peaks.

## Fix

1. Record which peaks had unidirectional liftover (for provenance)
2. Set `_det = 0` where `_orth = 0` (detection without orthology is meaningless)
3. Recalculate `n_species_det` and `species_detected`
4. Save corrected annotation as `master_annotation_corrected.tsv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# Paths
BASE_DIR = Path('/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3')
ANNO_FILE = BASE_DIR / '07_master_annotation' / 'master_annotation_final.tsv'
OUTPUT_DIR = BASE_DIR / '07_master_annotation'

SPECIES = ['Human', 'Bonobo', 'Chimpanzee', 'Gorilla', 'Macaque', 'Marmoset']

print(f'Loading master annotation from:\n  {ANNO_FILE}')
anno = pd.read_csv(ANNO_FILE, sep='\t', low_memory=False)
print(f'Loaded {len(anno):,} peaks x {len(anno.columns)} columns')

Loading master annotation from:
  /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv
Loaded 1,142,003 peaks x 53 columns


## 1. Verify peak widths (sanity check)

All peaks should be exactly 500 bp. Let's verify from the BED files.

In [2]:
import subprocess

print('=== Peak width verification (from 10_final BED files) ===')
print()
for sp in SPECIES:
    bed_file = BASE_DIR / '10_final' / f'all_peaks_{sp}.bed'
    if not bed_file.exists():
        print(f'  {sp}: BED file not found')
        continue
    bed = pd.read_csv(bed_file, sep='\t', header=None, names=['chr','start','end','peak_id'])
    widths = bed['end'] - bed['start']
    n_500 = (widths == 500).sum()
    n_other = (widths != 500).sum()
    print(f'  {sp:12s}: {len(bed):>10,} peaks  |  500bp: {n_500:>10,}  |  other: {n_other}')

# Also check the unified consensus in hg38
merged_bed = BASE_DIR / '02_merged_consensus' / 'unified_consensus_hg38_with_ids.bed'
if merged_bed.exists():
    bed = pd.read_csv(merged_bed, sep='\t', header=None)
    widths = bed.iloc[:, 2] - bed.iloc[:, 1]
    print(f'  {"hg38 consensus":12s}: {len(bed):>10,} peaks  |  500bp: {(widths==500).sum():>10,}  |  other: {(widths!=500).sum()}')

print()
print('✓ All peaks are 500bp — pipeline is consistent.')

=== Peak width verification (from 10_final BED files) ===

  Human       :  1,039,336 peaks  |  500bp:  1,039,336  |  other: 0
  Bonobo      :    992,889 peaks  |  500bp:    992,889  |  other: 0
  Chimpanzee  :  1,029,053 peaks  |  500bp:  1,029,053  |  other: 0
  Gorilla     :  1,013,198 peaks  |  500bp:  1,013,198  |  other: 0
  Macaque     :    982,905 peaks  |  500bp:    982,905  |  other: 0
  Marmoset    :    971,722 peaks  |  500bp:    971,722  |  other: 0
  hg38 consensus:  1,039,336 peaks  |  500bp:  1,039,336  |  other: 0

✓ All peaks are 500bp — pipeline is consistent.


## 2. Audit `_det` vs `_orth` discrepancy

Find all (peak, species) pairs where `_det=1` but `_orth=0`.

In [3]:
print('=== _det vs _orth discrepancy: det=1 AND orth=0 ===')
print()
print(f'{"Species":>12s}  {"det":>8s}  {"orth":>8s}  {"det_not_orth":>12s}  {"orth_not_det":>12s}')
print('-' * 60)

total_det_not_orth = 0
per_species_bad = {}

for sp in SPECIES:
    det  = anno[f'{sp}_det'].astype(int)
    orth = anno[f'{sp}_orth'].astype(int)
    
    det_not_orth = ((det == 1) & (orth == 0)).sum()
    orth_not_det = ((orth == 1) & (det == 0)).sum()
    
    per_species_bad[sp] = det_not_orth
    total_det_not_orth += det_not_orth
    
    print(f'{sp:>12s}  {det.sum():>8,}  {orth.sum():>8,}  {det_not_orth:>12,}  {orth_not_det:>12,}')

print('-' * 60)
print(f'Total (peak,species) pairs with det=1 & orth=0: {total_det_not_orth:,}')

# Unique peaks affected
mask_any_bad = pd.Series(False, index=anno.index)
for sp in SPECIES:
    mask_any_bad |= ((anno[f'{sp}_det'] == 1) & (anno[f'{sp}_orth'] == 0))
print(f'Unique peaks affected: {mask_any_bad.sum():,}')
print()

# Break down by peak type
print('Affected peaks by type:')
print(anno.loc[mask_any_bad, 'peak_type'].value_counts().to_string())
print()

# Verify: no coordinates exist when orth=0
print('Sanity check: do det=1 & orth=0 entries have coordinates?')
for sp in ['Bonobo', 'Chimpanzee', 'Gorilla', 'Macaque', 'Marmoset']:
    bad = anno[(anno[f'{sp}_det'] == 1) & (anno[f'{sp}_orth'] == 0)]
    has_coords = bad[f'{sp}_chr'].notna().sum()
    print(f'  {sp}: {len(bad)} bad entries, {has_coords} with coordinates (should be 0)')
print()

# Note: orth_not_det is EXPECTED and CORRECT
# It means the genomic region is conserved but the peak was not called
# (= regulatory divergence, not a data error)
print('Note: orth=1 & det=0 is CORRECT — it means regulatory divergence')
print('      (orthologous sequence exists, but no open chromatin peak was called)')

=== _det vs _orth discrepancy: det=1 AND orth=0 ===

     Species       det      orth  det_not_orth  orth_not_det
------------------------------------------------------------
       Human   629,491  1,039,336             0       409,845
      Bonobo   213,907   992,889         3,599       782,581
  Chimpanzee   419,227  1,029,053         3,083       612,909
     Gorilla   490,687  1,013,198         4,560       527,071
     Macaque   299,548   982,905         6,120       689,477
    Marmoset   477,942   971,722        19,565       513,345
------------------------------------------------------------
Total (peak,species) pairs with det=1 & orth=0: 36,927
Unique peaks affected: 33,156

Affected peaks by type:
peak_type
unified           31062
human_specific     2094

Sanity check: do det=1 & orth=0 entries have coordinates?
  Bonobo: 3599 bad entries, 0 with coordinates (should be 0)
  Chimpanzee: 3083 bad entries, 0 with coordinates (should be 0)
  Gorilla: 4560 bad entries, 0 with coordi

## 3. Understand the mechanism: unidirectional liftover

Count liftback stats from the pipeline BED files to confirm the numbers.

In [4]:
print('=== Pipeline liftback statistics ===')
print()
print(f'{"Species":>12s}  {"consensus":>10s}  {"liftback":>10s}  {"unmapped":>10s}  {"size_rej":>10s}  {"recip_pass":>10s}  {"recip_fail":>10s}  {"sp_specific":>12s}  {"10_final":>10s}')
print('-' * 120)

consensus_count = len(pd.read_csv(BASE_DIR / '02_merged_consensus' / 'unified_consensus_hg38_with_ids.bed', sep='\t', header=None))

for sp in ['Bonobo', 'Chimpanzee', 'Gorilla', 'Macaque', 'Marmoset']:
    # Raw liftback
    lb = len(pd.read_csv(BASE_DIR / '04_lifted_back' / f'unified_consensus_{sp}.bed', sep='\t', header=None))
    # Unmapped = failed liftback
    unmap_file = BASE_DIR / '04_lifted_back' / f'unified_consensus_{sp}.unmapped.bed'
    unmap = sum(1 for _ in open(unmap_file))  # includes header lines from liftOver
    # Size rejected
    size_rej = len(pd.read_csv(BASE_DIR / '04b_liftback_filtered' / f'unified_consensus_{sp}_size_rejected.bed', sep='\t', header=None))
    # Reciprocal pass/fail
    recip_pass = len(pd.read_csv(BASE_DIR / '04c_reciprocal' / f'{sp}_reciprocal_pass.bed', sep='\t', header=None))
    recip_fail = len(pd.read_csv(BASE_DIR / '04c_reciprocal' / f'{sp}_reciprocal_fail.bed', sep='\t', header=None))
    # Species-specific
    sp_spec = len(pd.read_csv(BASE_DIR / '05_species_specific' / f'{sp}_specific_peaks.bed', sep='\t', header=None))
    # Final
    final = len(pd.read_csv(BASE_DIR / '10_final' / f'all_peaks_{sp}.bed', sep='\t', header=None))
    
    print(f'{sp:>12s}  {consensus_count:>10,}  {lb:>10,}  {unmap:>10,}  {size_rej:>10,}  {recip_pass:>10,}  {recip_fail:>10,}  {sp_spec:>12,}  {final:>10,}')

print()
print('Key relationships:')
print('  _orth=1 ←→ peak has liftback coordinates (from 04_lifted_back)')
print('  _orth=0 ←→ peak is in the unmapped file (liftback failed)')
print('  _det=1  ←→ species name appears in species_detected column of 02_merged_consensus')
print()
print('The discrepancy: species→hg38 liftover succeeded (so species appears in species_detected),')
print('but hg38→species liftback FAILED (so _orth=0, no coordinates).')
print('This is a unidirectional liftover artefact.')

=== Pipeline liftback statistics ===

     Species   consensus    liftback    unmapped    size_rej  recip_pass  recip_fail   sp_specific    10_final
------------------------------------------------------------------------------------------------------------------------
      Bonobo   1,039,336     983,303      55,460         573     970,631      12,672         9,586     992,889
  Chimpanzee   1,039,336   1,020,701      17,316       1,319   1,007,840      12,861         8,352   1,029,053
     Gorilla   1,039,336     999,337      38,304       1,695     983,905      15,432        13,861   1,013,198
     Macaque   1,039,336     965,042      72,062       2,232     922,270      42,772        17,863     982,905
    Marmoset   1,039,336     918,717     115,085       5,537     866,837      51,880        53,005     971,722

Key relationships:
  _orth=1 ←→ peak has liftback coordinates (from 04_lifted_back)
  _orth=0 ←→ peak is in the unmapped file (liftback failed)
  _det=1  ←→ species name appe

## 4. Record unidirectional liftover peaks

Before correcting `_det`, we record which peaks had unidirectional liftover
for each species. This preserves the provenance information.

In [5]:
# Create a copy to modify
anno_corrected = anno.copy()

# Add per-species unidirectional liftover columns
# 1 = this species had a forward liftover to hg38 but the reverse liftback failed
for sp in SPECIES:
    col_name = f'{sp}_unidirectional_liftover'
    anno_corrected[col_name] = (
        (anno_corrected[f'{sp}_det'] == 1) & (anno_corrected[f'{sp}_orth'] == 0)
    ).astype(int)

# Summary column: comma-separated list of species with unidirectional liftover
def get_unidirectional_species(row):
    species_list = []
    for sp in SPECIES:
        if row[f'{sp}_unidirectional_liftover'] == 1:
            species_list.append(sp)
    return ','.join(species_list) if species_list else ''

anno_corrected['unidirectional_liftover_species'] = anno_corrected.apply(
    get_unidirectional_species, axis=1
)

n_unidirectional = (anno_corrected['unidirectional_liftover_species'] != '').sum()
print(f'Peaks with unidirectional liftover in >= 1 species: {n_unidirectional:,}')
print()

# Show per-species counts
print('Per-species unidirectional liftover counts:')
for sp in SPECIES:
    n = anno_corrected[f'{sp}_unidirectional_liftover'].sum()
    print(f'  {sp:12s}: {n:,}')

Peaks with unidirectional liftover in >= 1 species: 33,156

Per-species unidirectional liftover counts:
  Human       : 0
  Bonobo      : 3,599
  Chimpanzee  : 3,083
  Gorilla     : 4,560
  Macaque     : 6,120
  Marmoset    : 19,565


## 5. Correct `_det` columns

Set `_det = 0` wherever `_orth = 0`. Detection without orthology is meaningless
because:
- We have no species-native coordinates for this peak
- We cannot quantify reads at this locus in that species
- The "detection" was an artefact of the forward liftover overlapping the
  consensus window

In [6]:
print('=== Correcting _det columns ===')
print()

total_corrected = 0
for sp in SPECIES:
    mask = (anno_corrected[f'{sp}_det'] == 1) & (anno_corrected[f'{sp}_orth'] == 0)
    n_fix = mask.sum()
    total_corrected += n_fix
    anno_corrected.loc[mask, f'{sp}_det'] = 0
    print(f'  {sp:12s}: set {n_fix:,} entries from det=1 to det=0')

print(f'\nTotal corrections: {total_corrected:,}')

# Recalculate n_species_det
det_cols = [f'{sp}_det' for sp in SPECIES]
anno_corrected['n_species_det'] = anno_corrected[det_cols].sum(axis=1)

# Recalculate species_detected string
def rebuild_species_detected(row):
    detected = [sp for sp in SPECIES if row[f'{sp}_det'] == 1]
    return ','.join(detected) if detected else ''

anno_corrected['species_detected'] = anno_corrected.apply(rebuild_species_detected, axis=1)

print()
print('Recalculated n_species_det and species_detected.')

=== Correcting _det columns ===

  Human       : set 0 entries from det=1 to det=0
  Bonobo      : set 3,599 entries from det=1 to det=0
  Chimpanzee  : set 3,083 entries from det=1 to det=0
  Gorilla     : set 4,560 entries from det=1 to det=0
  Macaque     : set 6,120 entries from det=1 to det=0
  Marmoset    : set 19,565 entries from det=1 to det=0

Total corrections: 36,927

Recalculated n_species_det and species_detected.


## 6. Validate corrections

In [7]:
print('=== Post-correction validation ===')
print()

# Verify no more det=1 & orth=0
remaining_bad = 0
for sp in SPECIES:
    bad = ((anno_corrected[f'{sp}_det'] == 1) & (anno_corrected[f'{sp}_orth'] == 0)).sum()
    remaining_bad += bad
    if bad > 0:
        print(f'  ERROR: {sp} still has {bad} det=1 & orth=0 entries!')

if remaining_bad == 0:
    print('✓ No remaining det=1 & orth=0 entries. Fix is complete.')
print()

# Verify orth_not_det is unchanged (these are regulatory divergence, should stay)
print('Regulatory divergence (orth=1, det=0) — should be unchanged:')
for sp in SPECIES:
    orth_not_det_old = ((anno[f'{sp}_orth'] == 1) & (anno[f'{sp}_det'] == 0)).sum()
    orth_not_det_new = ((anno_corrected[f'{sp}_orth'] == 1) & (anno_corrected[f'{sp}_det'] == 0)).sum()
    status = '✓' if orth_not_det_old == orth_not_det_new else '✗ CHANGED'
    print(f'  {sp:12s}: {orth_not_det_old:>8,} → {orth_not_det_new:>8,}  {status}')

print()
print('n_species_det distribution (before vs after):')
print('  Before:')
print(anno['n_species_det'].value_counts().sort_index().to_string())
print()
print('  After:')
print(anno_corrected['n_species_det'].value_counts().sort_index().to_string())

=== Post-correction validation ===

✓ No remaining det=1 & orth=0 entries. Fix is complete.

Regulatory divergence (orth=1, det=0) — should be unchanged:
  Human       :  409,845 →  409,845  ✓
  Bonobo      :  782,581 →  782,581  ✓
  Chimpanzee  :  612,909 →  612,909  ✓
  Gorilla     :  527,071 →  527,071  ✓
  Macaque     :  689,477 →  689,477  ✓
  Marmoset    :  513,345 →  513,345  ✓

n_species_det distribution (before vs after):
  Before:
n_species_det
1    581802
2    189450
3    126355
4     97886
5     79569
6     66941

  After:
n_species_det
0     15052
1    573036
2    187841
3    125776
4     97719
5     78521
6     64058


## 7. Impact on peak categories

In [8]:
print('=== Impact on peak categories ===')
print()

# Compare before/after for human_specific peaks
hs_old = anno[anno['peak_type'] == 'human_specific']
hs_new = anno_corrected[anno_corrected['peak_type'] == 'human_specific']

print(f'Human-specific peaks: {len(hs_old):,}')
print(f'  Before: n_species_det distribution:')
print(f'    {hs_old["n_species_det"].value_counts().sort_index().to_dict()}')
print(f'  After:  n_species_det distribution:')
print(f'    {hs_new["n_species_det"].value_counts().sort_index().to_dict()}')
print()

# The 2,094 human_specific peaks that previously had det in other species
# should now show det only in Human (or 0 if Human_det was 0)
hs_previously_multi = hs_old[hs_old['n_species_det'] > 1]
hs_now_fixed = anno_corrected.loc[hs_previously_multi.index]
print(f'Human-specific peaks that previously showed det>1: {len(hs_previously_multi):,}')
print(f'  Their corrected n_species_det: {hs_now_fixed["n_species_det"].value_counts().sort_index().to_dict()}')
print()

# Show impact on unified peaks
uni = anno_corrected[anno_corrected['peak_type'] == 'unified']
print(f'Unified peaks: {len(uni):,}')
print(f'  n_species_det distribution after correction:')
for n in sorted(uni['n_species_det'].unique()):
    count = (uni['n_species_det'] == n).sum()
    print(f'    {n}: {count:>8,}')

=== Impact on peak categories ===

Human-specific peaks: 3,202
  Before: n_species_det distribution:
    {1: 2339, 2: 422, 3: 219, 4: 121, 5: 65, 6: 36}
  After:  n_species_det distribution:
    {0: 1608, 1: 1594}

Human-specific peaks that previously showed det>1: 863
  Their corrected n_species_det: {0: 377, 1: 486}

Unified peaks: 1,036,134
  n_species_det distribution after correction:
    0:   13,444
    1:  468,775
    2:  187,841
    3:  125,776
    4:   97,719
    5:   78,521
    6:   64,058


## 8. Save corrected annotation

In [9]:
# Reorder columns: keep original order, then append new columns at the end
original_cols = list(anno.columns)
new_cols = [f'{sp}_unidirectional_liftover' for sp in SPECIES] + ['unidirectional_liftover_species']
all_cols = original_cols + new_cols
anno_corrected = anno_corrected[all_cols]

# Save
out_file = OUTPUT_DIR / 'master_annotation_corrected.tsv'
anno_corrected.to_csv(out_file, sep='\t', index=False)
print(f'Saved corrected annotation to:\n  {out_file}')
print(f'  {len(anno_corrected):,} peaks x {len(anno_corrected.columns)} columns')
print()

# Also save a summary of unidirectional liftover peaks
unidirectional_peaks = anno_corrected[anno_corrected['unidirectional_liftover_species'] != ''][
    ['peak_id', 'peak_type', 'n_species_det', 'n_species_orth', 'unidirectional_liftover_species']
]
unidirectional_file = OUTPUT_DIR / 'unidirectional_liftover_peaks.tsv'
unidirectional_peaks.to_csv(unidirectional_file, sep='\t', index=False)
print(f'Saved unidirectional liftover peak list to:\n  {unidirectional_file}')
print(f'  {len(unidirectional_peaks):,} peaks with unidirectional liftover')

Saved corrected annotation to:
  /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/master_annotation_corrected.tsv
  1,142,003 peaks x 60 columns

Saved unidirectional liftover peak list to:
  /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/unidirectional_liftover_peaks.tsv
  33,156 peaks with unidirectional liftover


## 9. Summary

### What changed

| Column | Change |
|--------|--------|
| `{Species}_det` | Set to `0` where `{Species}_orth = 0` (36,927 corrections) |
| `n_species_det` | Recalculated from corrected `_det` columns |
| `species_detected` | Rebuilt from corrected `_det` columns |

### New columns added

| Column | Description |
|--------|-------------|
| `{Species}_unidirectional_liftover` | `1` if species→hg38 liftover succeeded but hg38→species liftback failed |
| `unidirectional_liftover_species` | Comma-separated list of species with unidirectional liftover |

### Column semantics (corrected)

| Column | Meaning |
|--------|--------|
| `_orth = 1` | Orthologous genomic region exists (bidirectional liftover succeeded) |
| `_orth = 0` | No orthologous region (liftback failed) |
| `_det = 1` | Peak was called AND orthologous region exists (orth=1 guaranteed) |
| `_det = 0, _orth = 1` | Regulatory divergence: sequence conserved, but no open chromatin |
| `_det = 0, _orth = 0` | No orthologous region |
| `_unidirectional_liftover = 1` | Forward lift worked, reverse failed (archived provenance) |

In [10]:
print('=== Final Summary ===')
print()
print('Files produced:')
print(f'  1. {OUTPUT_DIR / "master_annotation_corrected.tsv"}')
print(f'  2. {OUTPUT_DIR / "unidirectional_liftover_peaks.tsv"}')
print()
print('The original master_annotation_final.tsv is UNCHANGED.')
print('Downstream notebooks should load master_annotation_corrected.tsv instead.')
print()
print('Total corrections: 36,927 (peak,species) pairs across 33,156 unique peaks')
print('  - 31,062 unified peaks (had det=1 in a species where liftback failed)')
print('  - 2,094 human_specific peaks (showed det in NHP species; impossible)')

=== Final Summary ===

Files produced:
  1. /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/master_annotation_corrected.tsv
  2. /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/unidirectional_liftover_peaks.tsv

The original master_annotation_final.tsv is UNCHANGED.
Downstream notebooks should load master_annotation_corrected.tsv instead.

Total corrections: 36,927 (peak,species) pairs across 33,156 unique peaks
  - 31,062 unified peaks (had det=1 in a species where liftback failed)
  - 2,094 human_specific peaks (showed det in NHP species; impossible)
